# Retail Orders — Data Profiling & Quality Checks

This notebook profiles the supplied raw retail orders CSV against the supplied data dictionary. It reports completeness, uniqueness, validity, consistency, and freshness. It deliberately **does not silently repair** business data. Review the exceptions and obtain owner approval before publishing KPIs.

**Run:** Put `retail-orders-raw.csv` and `retail-data-dictionary.csv` in the same folder as this notebook, or update the file paths below.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date

ORDERS_FILE = Path("retail-orders-raw.csv")
DICTIONARY_FILE = Path("retail-data-dictionary.csv")

raw = pd.read_csv(ORDERS_FILE, dtype={"order_id": "string"})
dictionary = pd.read_csv(DICTIONARY_FILE)
print(f"Rows: {len(raw):,} | Columns: {len(raw.columns)}")
display(raw.head())
display(dictionary)


## 1. Normalize for testing (retain raw data unchanged)

In [ ]:
df = raw.copy()

# Normalize category-like text without changing the source file.
for col in ["customer_segment", "city", "category", "payment_status"]:
    if col in df:
        df[col] = df[col].astype("string").str.strip()

df["customer_segment_norm"] = df["customer_segment"].str.title()
df["category_norm"] = df["category"].str.title()
df["payment_status_norm"] = df["payment_status"].str.title()

# Parse dates using mixed-format support where available; day-first handles 03/01/2026 as 3 Jan.
df["order_date_parsed"] = pd.to_datetime(df["order_date"], errors="coerce", dayfirst=True)
df["quantity_num"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price_num"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["discount_num"] = pd.to_numeric(df["discount_pct"], errors="coerce")

ALLOWED_SEGMENTS = {"Student", "Fresher", "Professional"}
ALLOWED_CATEGORIES = {"Learning Kit", "Course Access", "Mentor Session"}
ALLOWED_STATUSES = {"Paid", "Pending", "Failed", "Refunded"}
TODAY = pd.Timestamp.today().normalize()


## 2. Executable row-level quality checks

In [ ]:
checks = pd.DataFrame(index=df.index)
checks["order_id_present"] = df["order_id"].notna() & df["order_id"].str.strip().ne("")
checks["order_date_valid"] = (
    df["order_date_parsed"].notna()
    & df["order_date_parsed"].between(pd.Timestamp("2025-01-01"), TODAY)
)
checks["segment_valid"] = df["customer_segment_norm"].isin(ALLOWED_SEGMENTS)
checks["city_present"] = df["city"].notna() & df["city"].str.strip().ne("")
checks["category_valid"] = df["category_norm"].isin(ALLOWED_CATEGORIES)
checks["quantity_positive_integer"] = (
    df["quantity_num"].notna()
    & (df["quantity_num"] > 0)
    & np.isclose(df["quantity_num"].fillna(0) % 1, 0)
)
checks["unit_price_nonnegative"] = df["unit_price_num"].notna() & (df["unit_price_num"] >= 0)
checks["discount_valid_or_missing"] = df["discount_num"].isna() | df["discount_num"].between(0, 100)
checks["payment_status_valid"] = df["payment_status_norm"].isin(ALLOWED_STATUSES)

# Duplicate ID check: every occurrence of a repeated ID is flagged for review.
checks["order_id_unique"] = ~df["order_id"].duplicated(keep=False)

# Missing discount is not automatically valid for KPI use: it requires owner clarification.
checks["discount_present_for_revenue"] = df["discount_num"].notna()

display(checks)


## 3. Profile dimensions: completeness, uniqueness, validity, consistency, freshness

In [ ]:
completeness = (raw.notna() & raw.astype("string").apply(lambda s: s.str.strip().ne(""))).mean().mul(100)
uniqueness = raw.nunique(dropna=True).div(len(raw)).mul(100)

validity = pd.Series({
    "order_date": checks["order_date_valid"].mean() * 100,
    "customer_segment": checks["segment_valid"].mean() * 100,
    "category": checks["category_valid"].mean() * 100,
    "quantity": checks["quantity_positive_integer"].mean() * 100,
    "unit_price": checks["unit_price_nonnegative"].mean() * 100,
    "discount_pct (range only)": checks["discount_valid_or_missing"].mean() * 100,
    "payment_status": checks["payment_status_valid"].mean() * 100,
})
consistency = pd.Series({
    "unique order_id": checks["order_id_unique"].mean() * 100,
    "all row checks pass (excluding missing discount)": checks.drop(columns=["discount_present_for_revenue"]).all(axis=1).mean() * 100,
})
latest_date = df["order_date_parsed"].max()
freshness_days = (TODAY - latest_date.normalize()).days if pd.notna(latest_date) else np.nan

print("Completeness (% non-empty by column)")
display(completeness.rename("completeness_pct").to_frame())
print("Uniqueness (% distinct non-null values by column; order_id uniqueness is separately tested)")
display(uniqueness.rename("distinct_pct").to_frame())
print("Validity (%)"); display(validity.rename("valid_pct").to_frame())
print("Consistency (%)"); display(consistency.rename("pass_pct").to_frame())
print({"latest_valid_order_date": latest_date, "age_in_days": freshness_days})


## 4. Exceptions and quarantine view

In [ ]:
checks["row_passes_critical_rules"] = checks[
    ["order_id_present", "order_date_valid", "segment_valid", "city_present",
     "category_valid", "quantity_positive_integer", "unit_price_nonnegative",
     "discount_valid_or_missing", "payment_status_valid", "order_id_unique"]
].all(axis=1)

exceptions = df.loc[~checks["row_passes_critical_rules"]].copy()
exceptions["failed_checks"] = checks.loc[~checks["row_passes_critical_rules"]].apply(
    lambda row: ", ".join(row.index[~row]), axis=1
)
display(exceptions[["order_id", "order_date", "quantity", "discount_pct", "payment_status", "failed_checks"]])

# Exact duplicate records are candidates for removal, not automatic deletions.
exact_duplicate_rows = raw[raw.duplicated(keep=False)].sort_values(list(raw.columns))
print(f"Exact duplicate rows flagged for owner review: {len(exact_duplicate_rows)}")
display(exact_duplicate_rows)


## 5. Illustrative KPI calculation after strict validation

In [ ]:
# Strict eligible rows: all critical checks pass AND discount is present.
eligible = df.loc[checks["row_passes_critical_rules"] & checks["discount_present_for_revenue"]].copy()
eligible = eligible.drop_duplicates(subset=["order_id"], keep="first")  # only after exact-duplicate review/approval
eligible["net_order_value"] = (
    eligible["quantity_num"] * eligible["unit_price_num"] *
    (1 - eligible["discount_num"] / 100)
)
paid = eligible.loc[eligible["payment_status_norm"].eq("Paid")]

paid_orders = paid["order_id"].nunique()
net_sales = paid["net_order_value"].sum()
kpi_snapshot = pd.Series({
    "Paid Orders": paid_orders,
    "Net Sales (INR)": net_sales,
    "Average Order Value (INR)": net_sales / paid_orders if paid_orders else np.nan,
    "Units Sold": paid["quantity_num"].sum(),
    "Average Discount (%)": paid["discount_num"].mean(),
})
display(kpi_snapshot.to_frame("value"))
print("Important: this is a small deliberately imperfect training sample, not a production KPI baseline.")


## 6. Export profile results

In [ ]:
profile = pd.DataFrame({
    "completeness_pct": completeness,
    "distinct_pct": uniqueness
})
profile.to_csv("retail_data_profile_summary.csv", index_label="column")
checks.to_csv("retail_row_quality_checks.csv", index_label="source_row")
print("Wrote retail_data_profile_summary.csv and retail_row_quality_checks.csv")
